In [ ]:
# PPO Evaluation 

import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.append('..')
from environment.trading_env import TradingEnv
from experts.ppo_expert import PPOAgent

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
ASSET = "BTCUSDT"
CONFIG_PATH = "../configs/experts.yaml"
TEST_DATA_PATH = f"../data/processed/{ASSET}/train.csv"
MODEL_PATH = f"../checkpoints/{ASSET}/no_diversity/PPO/ppo_expert.pt"

# Load config (for hyperparameters)
with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)
ppo_cfg = cfg["ppo"]

In [ ]:
# ------------------------------------------------------------------
# 1. Load test data
# ------------------------------------------------------------------
test_df = pd.read_csv(TEST_DATA_PATH, index_col=0, parse_dates=True)
print(f"Test data shape: {test_df.shape}")

# ------------------------------------------------------------------
# 2. Load checkpoint
# ------------------------------------------------------------------
ckpt = torch.load(MODEL_PATH, map_location='cpu')

if "network" in ckpt:
    state_dict = ckpt["network"]
    saved_features = ckpt.get("features", None)
    saved_n_state = ckpt.get("n_state_features", None)
else:
    state_dict = ckpt
    saved_features = None
    saved_n_state = None

# ------------------------------------------------------------------
# 3. Infer architecture from the state_dict
# ------------------------------------------------------------------
# Find all actor trunk weight keys
actor_weight_keys = [k for k in state_dict.keys() if k.startswith("actor_trunk.") and k.endswith(".weight")]
if not actor_weight_keys:
    # Fallback to legacy naming if needed
    actor_weight_keys = [k for k in state_dict.keys() if k.startswith("trunk.") and k.endswith(".weight")]

if not actor_weight_keys:
    raise KeyError("No trunk weight keys found in state_dict. Is this a PPOAgent checkpoint?")

# Determine number of layers
# Each linear layer has a weight key. The sequential has linear and activation alternating.
# We count the number of linear layers = number of weight keys.
num_layers = len(actor_weight_keys)

# Determine hidden dimension from first layer's output size
first_weight = state_dict[actor_weight_keys[0]]
hidden_dim = first_weight.shape[0]   # output dimension of first linear

# Determine state dimension from first layer's input size
state_dim_from_weights = first_weight.shape[1]

# Determine action dimension from actor head
if "actor_head.weight" in state_dict:
    action_dim = state_dict["actor_head.weight"].shape[0]
else:
    # Fallback: assume 3 (discrete actions)
    action_dim = 3

print(f"Inferred architecture: hidden_dim={hidden_dim}, num_layers={num_layers}, "
      f"action_dim={action_dim}, state_dim={state_dim_from_weights}")

# ------------------------------------------------------------------
# 4. Determine number of environment state features
# ------------------------------------------------------------------
n_state_features = saved_n_state if saved_n_state is not None else 6
expected_features = state_dim_from_weights - n_state_features
print(f"Environment state features (assumed): {n_state_features}")
print(f"Checkpoint expects {expected_features} feature columns.")

# ------------------------------------------------------------------
# 5. Select feature columns (same as before)
# ------------------------------------------------------------------
if saved_features is not None and len(saved_features) == expected_features:
    features = saved_features
    print("Using features from checkpoint:", features)
else:
    config_features = ppo_cfg.get("features", []) if 'ppo_cfg' in locals() else []
    if len(config_features) == expected_features:
        features = config_features
        print("Using features from config:", features)
    else:
        print(f"Config has {len(config_features)} features, but checkpoint expects {expected_features}.")
        exclude = ["open", "high", "low", "close", "volume", "log_volume", "returns", "return_5"]
        candidates = [c for c in test_df.columns if c not in exclude]
        if len(candidates) >= expected_features:
            features = candidates[:expected_features]
        else:
            features = test_df.columns[:expected_features].tolist()
        print("Using auto‑selected features:", features)

# ------------------------------------------------------------------
# 6. Create environment with the selected features
# ------------------------------------------------------------------
env_params = {
    "continuous_actions": False,
    "leverage": 1,
    # Add other env parameters if needed
}
env = TradingEnv(df=test_df, feature_cols=features, expert_id=1, **env_params)

# Verify that the computed state dimension matches the network's input
state_dim = len(features) + env._n_state_features
if state_dim != state_dim_from_weights:
    print(f"WARNING: Computed state_dim ({state_dim}) does not match expected ({state_dim_from_weights}).")
    # If they differ, you may need to adjust n_state_features manually

# ------------------------------------------------------------------
# 7. Instantiate the agent using the inferred architecture
# ------------------------------------------------------------------
agent = PPOAgent(
    state_dim=state_dim,          # must match state_dim_from_weights
    action_dim=action_dim,
    # Use the architecture extracted from the checkpoint
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    activation="tanh",            # common default; if different, you can try to infer
    # Hyperparameters below are not used during inference, but required by __init__
    lr=1e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_eps=0.2,
    entropy_coef=0.01,
    value_coef=0.5,
    max_grad_norm=0.5,
    update_epochs=10,
    batch_size=256,
    rollout_steps=2048,
)

# Load the network state dictionary
agent.network.load_state_dict(state_dict)
print("Model loaded successfully. Ready for inference with agent.act(state).")


In [ ]:
# Environment
asset_type = "crypto" if "USDT" in ASSET else "forex"
window_size = cfg.get("window_size", 20)
env_params = {
    "df": test_df,
    "feature_cols": features,
    "expert_id": 1,
    "asset_type": asset_type,
    "contract_size": 1.0 if asset_type == "crypto" else 100000.0,
    "commission_rate": 0.0005 if asset_type == "crypto" else 1.5/100000.0,
    "spread_rate": 0.0002 if asset_type == "crypto" else 0.00005,
    "slippage_rate": 0.0001 if asset_type == "crypto" else 0.00001,
    "leverage": 5 if asset_type == "crypto" else 30,
    "maintenance_margin_ratio": 0.5,
    "initial_balance": 10000.0,
    "window_size": window_size,
    "reward_scale": 1.0,
    "clip_reward": (-10.0, 10.0),
    "max_position_fraction": 0.95,
    "continuous_actions": False,
}
env = TradingEnv(**env_params)

# PPO Agent
state_dim = len(features) + 6
agent = PPOAgent(
    state_dim=state_dim,
    action_dim=3,
    entropy_coef=0,
    hidden_dim=ppo_cfg.get("hidden_dim", 256),
    num_layers=ppo_cfg.get("num_layers", 2),
    activation=ppo_cfg.get("activation", "tanh"),
    log_dir=None,
    device="cpu",
)

# Load weights (strict=False to ignore optimizer keys)
agent.network.load_state_dict(state_dict, strict=False)
print("PPO agent loaded.")

In [ ]:
obs, _ = env.reset()
done = False
equity_curve = [env.initial_balance]

while not done:
    # Unpack the three outputs from act()
    action, log_prob, value = agent.act(obs)
    
    obs, reward, terminated, truncated, info = env.step(action)
    equity_curve.append(info.get('equity', env.initial_balance))
    
    done = terminated or truncated

equity_curve = np.array(equity_curve)
cum_return = (equity_curve - equity_curve[0]) / equity_curve[0]

print(f"Final equity: {equity_curve[-1]:.2f}")
print(f"Final cumulative return: {cum_return[-1]:.4f} ({cum_return[-1]*100:.2f}%)")

In [ ]:
# Hourly returns (percentage change of equity)
hourly_returns = np.diff(equity_curve) / equity_curve[:-1]
total_ret = cum_return[-1]
n = len(hourly_returns)

# Annualisation factor – adjust to your market:
#   Crypto (24/7/365)  → 8760
#   Forex (24/5)        → ~6240 (24 * 260 trading days)
hours_per_year = 8760   # <-- change if necessary

ann_ret = (1 + total_ret) ** (hours_per_year / n) - 1 if n > 0 else 0.0
ann_vol = np.std(hourly_returns) * np.sqrt(hours_per_year)
sharpe = ann_ret / ann_vol if ann_vol != 0 else 0.0

# Drawdown (based on equity)
peak = np.maximum.accumulate(equity_curve)
drawdown = (peak - equity_curve) / peak
max_dd = np.max(drawdown)

win_rate = np.mean(hourly_returns > 0)

print("\n--- Performance (Hourly) ---")
print(f"Total Return:       {total_ret:.2%}")
print(f"Annualized Return:  {ann_ret:.2%}")
print(f"Annualized Vol:     {ann_vol:.2%}")
print(f"Sharpe Ratio:       {sharpe:.2f}")
print(f"Max Drawdown:       {max_dd:.2%}")
print(f"Win Rate:           {win_rate:.2%}")

In [ ]:
# Equity curve (cumulative return)
plt.figure(figsize=(14,6))
plt.plot(cum_return, label='PPO', linewidth=2)
plt.title("PPO Equity Curve (Cumulative Return)")
plt.xlabel("Step")
plt.ylabel("Cumulative Return")
plt.grid(True)
plt.legend()
plt.show()

# Drawdown
plt.figure(figsize=(14,3))
plt.fill_between(range(len(drawdown)), 0, drawdown, color='red', alpha=0.4)
plt.plot(drawdown, color='darkred', linewidth=0.5)
plt.title("Drawdown")
plt.xlabel("Step")
plt.ylabel("Drawdown")
plt.grid(True)
plt.show()